In [ ]:
print('Menu Detector')

In [ ]:
# import libraries

from google.colab import drive
import os
from collections import Counter
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models

In [ ]:
drive.mount('/content/drive')

In [ ]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print("Dataset Path:", DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping | class consolidation
    'cheesecake': 'dessert', # label grouping | class consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']

CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print("Classes:", CLASSES)
print("Class to Index Mapping:", CLASS_TO_IDX)

In [ ]:
# Define Transforms

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    normalize
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize
])

In [ ]:
# Custom Dataset

class FoodDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        for folder, label in CUSTOM_CLASS_MAPPING.items():
            folder_path = os.path.join(root, folder)
            if not os.path.isdir(folder_path):
                print("Missing folder:", folder_path)
                continue
            for f in sorted(os.listdir(folder_path)):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(folder_path, f), CLASS_TO_IDX[label]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# Train / Validation Split

torch.manual_seed(42)

train_full = FoodDataset(DATASET_PATH, transform=train_transform)
val_full = FoodDataset(DATASET_PATH, transform=val_transform)

indices = torch.randperm(len(train_full)).tolist()
split = int(0.8 * len(indices))

train_dataset = Subset(train_full, indices[:split])
val_dataset = Subset(val_full, indices[split:])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, num_workers=2)

counts = Counter(label for _, label in train_full.samples)
print("Images per class:", {CLASSES[i]: counts[i] for i in range(NUM_CLASSES)})
print("Train:", len(train_dataset), "| Val:", len(val_dataset))

In [ ]:
# Build Model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False  # freeze pretrained backbone

model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)  # new trainable head
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [ ]:
# Train Model

def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

MODEL_PATH = os.path.join(DATASET_PATH, 'menu_detector.pt')
EPOCHS = 10
best_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss {train_loss:.3f} acc {train_acc:.2%} | val loss {val_loss:.3f} acc {val_acc:.2%}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({'state_dict': model.state_dict(), 'classes': CLASSES}, MODEL_PATH)

print(f"Best val acc: {best_acc:.2%}")

In [ ]:
# Predict

def predict(image_path):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    image = val_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = model(image).softmax(1)[0]
    for name, p in sorted(zip(checkpoint['classes'], probs.tolist()), key=lambda t: -t[1]):
        print(f"{name:10s} {p:.2%}")

# predict('/content/my_plate.jpg')